<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/creating_your_own_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests

In [ ]:
import requests
url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)
response.status_code

In [ ]:
response.json()

In [ ]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import requests

def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=4_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    # GitHub API pagination starts at page 1
    for page in tqdm(range(1, num_pages + 1)):
        query = f"issues?page={page}&per_page={per_page}&state=all"
        response = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)

        if response.status_code != 200:
            print(f"\nError fetching page {page}: {response.status_code} - {response.text}")
            break

        issues = response.json()
        if isinstance(issues, list):
            batch.extend(issues)
        else:
            print(f"\nUnexpected response structure: {issues}")
            break

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    if all_issues:
        df = pd.DataFrame.from_records(all_issues)
        dest_file = issues_path / f"{repo}-issues.jsonl"
        df.to_json(dest_file, orient="records", lines=True)
        print(
            f"\nDownloaded {len(all_issues)} issues for {repo}! Dataset stored at {dest_file}"
        )
    else:
        print("\nNo issues were downloaded.")

In [ ]:
fetch_issues()

In [ ]:
from datasets import load_dataset
issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
issues_dataset

In [ ]:
def get_comments(issue_number):
  url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
  response = requests.get(url, headers=headers)

In [ ]:
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
issues_with_comments_dataset.push_to_hub("github-issues")